# Phase 2 on Colab — main experiment + false-negative audit

Sequencer for [`PHASE2_PLAN.md`](../PHASE2_PLAN.md). This notebook is deliberately thin: experiment logic belongs in the tested `embeded/` Python modules, not duplicated in notebook cells.

> **Current status (2026-09-27):** gate rule D1 is frozen and `embeded.train` / `embeded.evaluate` are implemented and unit-tested offline. **No Phase 2 training has been run yet** — the cells below still need a T4, regenerated Phase 1 artifacts and the scored blind audit before they produce results.

## Order of work
1. Start a Colab **T4 GPU** runtime, clone the code version containing the Phase 2 implementation, configure optional Hugging Face artifact sync, and verify the environment.
2. Run tests and inspect/retrieve the frozen Phase 1 artifacts. Resolve the G1 rule explicitly; do not bypass the current absolute-margin gate or erase its historical FAIL.
3. Run the blind C2/C3 50+50 audit, label the pairs without opening the key, and score it before training.
4. Run a tiny end-to-end training smoke test.
5. Run C0 once and C1/C2/C3 for seeds 13, 14, 15. Checkpoint outputs after each stage.
6. Review validation-selected thresholds and test results, create the Phase 2 table, and record the actual work in `PHASE2_PLAN.md` and `report/measurements.md`.

Never use the test split to tune a threshold or hyperparameter. Keep Hugging Face tokens in Colab Secrets, not notebook code.

## 0. Runtime and code checkout

In Colab choose **Runtime → Change runtime type → T4 GPU**. Set `REPO_REF` below to the branch/commit that contains the approved Phase 2 code. This notebook defaults to the Arena work branch; use `main` after the changes are merged.

In [ ]:
import os, subprocess

REPO_URL = 'https://github.com/ksu-gitreaper807/EmbedEd.git'
REPO_REF = 'arena/01a0e257-embeded'  # branch carrying the D1 gate rule + the Phase 2 runner; use main once merged
ROOT = '/content/EmbedEd'
if os.path.isdir(os.path.join(ROOT, '.git')):
    subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_REF, 'FETCH_HEAD'], cwd=ROOT, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_REF, '--single-branch', REPO_URL, ROOT], check=True)
%cd /content/EmbedEd
print('Checked out:', subprocess.check_output(['git','rev-parse','--short','HEAD'], text=True).strip())


## 1. Configure caches, install pinned dependencies, and verify GPU

Do not install/pin PyTorch separately: use the CUDA-matched build supplied by Colab. The cache pull is optional; without a configured HF dataset repo this session starts with local artifacts only.

In [ ]:
import os, sys
ROOT = '/content/EmbedEd'
ART = '/content/embeded-artifacts'
os.makedirs(ART, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf-home'
os.environ['EMBEDED_ARTIFACTS'] = ART
os.environ['EMBEDED_REPORT'] = ROOT + '/report/measurements.md'
try:
    from google.colab import userdata
    for key in ('EMBEDED_HF_REPO_ID', 'EMBEDED_HF_REPO_TYPE', 'EMBEDED_HF_REVISION', 'EMBEDED_HF_SUBDIR', 'HF_TOKEN'):
        try: value = userdata.get(key)
        except Exception: value = None
        if value: os.environ.setdefault(key, value)
except Exception:
    pass
%pip install -q -U transformers==4.46.3 datasets==2.20.0 sentence-transformers==3.0.1 rank-bm25==0.2.2 umap-learn==0.5.6 scikit-learn==1.6.1 gradio==5.49.1 pytest==8.3.2
import torch, transformers, datasets, sentence_transformers, sklearn
print(f'torch={torch.__version__} cuda={torch.cuda.is_available()} | transformers={transformers.__version__} | datasets={datasets.__version__} | sklearn={sklearn.__version__}')
if not torch.cuda.is_available(): raise RuntimeError('GPU is required. Select a Colab T4 runtime before continuing.')
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'T4' in gpu.upper(), f'Expected the measured T4 configuration; got {gpu}. Re-measure throughput before using the frozen budget.'
from scripts.hf_artifacts import main as hf_main
hf_main(['pull', '--if-configured'])

## 2. Preflight: artifacts, tests, and gate G1 under decision D1

Order matters: the gate reads the Phase 1 artifacts, so they are checked first — a fresh VM has none and `hardness_diagnostics` would otherwise die inside `numpy` with a bare `FileNotFoundError`.

Gate G1 now uses the **owner-approved scale-aware rule (decision D1, recorded 2026-09-27)**: `C1 < C2 ≤ C3` **and** standardised gap `d(C1→C2) ≥ HARDNESS_D_MIN = 0.5`, with the corpus percentiles reported alongside. The legacy `HARDNESS_MARGIN = 0.02` reading is still printed and recorded, but it is no longer the verdict; the 2026-09-26 FAIL stays verbatim in `report/measurements.md`. Do not edit `HARDNESS_D_MIN` to get past a FAIL — a failed manipulation means stop and fix the mining (SCOPE P6-3).

In [ ]:
import subprocess, sys
from embeded import settings as S

# 1. artifacts first: the gate and the diagnostics both need them, and the
#    message below is the one the modules print too (embeded.hardcheck)
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy'),
            S.artifact('corpus_emb.meta.json')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1', 'C2', 'C3')]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        'Phase 1 artifacts missing — regenerate all of them before the gate (PHASE2_PLAN §2 item 2):\n  '
        + '\n  '.join(missing)
        + '\n  python -m embeded.data.prepare_data --hf --verify-spec'
          '\n  python -m embeded.mining.semantic_index   # GPU encode, ~4 min on a T4'
          '\n  python -m embeded.negatives               # all three conditions together, ~15-30 min')

# 2. offline tests, then the scale-of-the-space view, then the gate
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'embeded/tests'], check=True)
subprocess.run([sys.executable, '-m', 'scripts.hardness_diagnostics', '--examples', '3'], check=True)
subprocess.run([sys.executable, '-m', 'embeded.hardcheck'], check=True)   # exit 1 = FAIL => stop
hf_main(['push', '--if-configured', '--include-report'])
print('G1 PASS under D1 — this run was appended to the G1 history next to the 2026-09-26 FAIL.')


### Artifact check

Only continue if the exact frozen corpus/triples are present and verified for the current code/settings version. Regenerate/retrieve them with the Phase 0/1 notebook if missing or stale. If anything about mining changes, regenerate **all three conditions together**, then rerun tests, gate, and diagnostics.

In [ ]:
from pathlib import Path
from embeded import settings as S
required = [S.artifact('fragments.jsonl'), S.artifact('corpus_emb.npy')] + [S.artifact(f'triples_{c}.jsonl') for c in ('C1','C2','C3')]
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError('Missing Phase 1 artifacts; restore/regenerate and validate them before training:\n' + '\n'.join(missing))
print('Artifacts present for VERSION', S.VERSION)
print('NOTE: confirm metadata/hashes and the shared anchors/positives/k; file existence alone is not an identity check.')

## 3. Make and complete the blind false-negative audit (50 C2 + 50 C3)

Generate the shuffled, condition-blind sample. Label each pair `clone`, `not_clone`, or `unsure` by functional equivalence. Do not inspect `audit_key.csv` until scoring. Download only the pair sheet and blank labels; keep the key private. After labeling, upload the completed CSV in the next cell. This audit is a Phase 2 entry criterion, not a result to fill in from token-Jaccard proxies.

In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import files
audit_dir = Path(ART) / 'audit'
pairs_file, labels_file, key_file = (audit_dir / n for n in ('audit_pairs.md', 'audit_labels.csv', 'audit_key.csv'))
if not (pairs_file.exists() and labels_file.exists() and key_file.exists()):
    subprocess.run([sys.executable, '-m', 'scripts.audit_sample', 'make', '--n', '50', '--seed', '13'], check=True)
else:
    print('Existing audit artifacts found; reusing them to avoid overwriting labels.')
hf_main(['push', '--if-configured'])  # checkpoint the blind key + blank sheet before labeling
files.download(str(pairs_file))
files.download(str(labels_file))
print('Do not download/open audit_key.csv until after labels are final.')


### After labeling offline: upload the completed labels and score

Upload the filled `audit_labels.csv` only. Keep `audit_key.csv` unchanged and do not edit it by hand. Scoring writes the blinded C2/C3 rates, Wilson intervals, unsure upper bounds, JSON result, and report section. Review the audit result and document how it affects interpretation before allowing training.

In [ ]:
from google.colab import files
uploaded = files.upload()
if 'audit_labels.csv' not in uploaded: raise ValueError('Upload the completed file named audit_labels.csv')
from pathlib import Path
import subprocess, sys
(Path(ART) / 'audit').mkdir(parents=True, exist_ok=True)
(Path(ART) / 'audit' / 'audit_labels.csv').write_bytes(uploaded['audit_labels.csv'])
subprocess.run([sys.executable, '-m', 'scripts.audit_sample', 'score'], check=True)
hf_main(['push', '--if-configured', '--include-report'])


## 4. Smoke test the Phase 2 runner

`embeded.train` and `embeded.evaluate` are implemented and unit-tested offline. `--smoke` runs the whole harness on a tiny subset — load → forward/backward → checkpoint save/reload (verified tensor-by-tensor) → validation scoring → test metrics — and is a harness check, never an experiment result. Full runs use the frozen recipe in `settings.py` (`MAX_LEN = 512`, `BATCH = 8`, `TRAIN_PAIRS_CAP = 32,000`, `EPOCHS = 1`, fp16 autocast).

In [ ]:
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('Phase 2 runner modules are missing from this checkout — check REPO_REF in cell 0.')
!python -m embeded.train --condition C1 --seed 13 --smoke
# trains a few steps, reloads the checkpoint, then evaluates on a tiny subset of
# both splits. Expect ~3 min on a T4; the numbers it prints mean nothing.

## 5. Evaluate the untuned C0 baseline once

C0 uses the untouched pretrained encoder and the same tokenizer, mean pooling, max length, and scoring path as the trained runs. Select any F1 threshold on validation only, apply it to test once, and retain predictions/configuration. Do not fine-tune C0.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'embeded.evaluate', '--condition', 'C0', '--seed', '13'], check=True)


## 6. Run the nine main training jobs (3 conditions × 3 seeds)

This sequential loop is intentionally fail-fast. It assumes the implementation follows the Phase 2 CLI contract in `PHASE2_PLAN.md`. Use one explicit-triplet loss and identical non-condition settings in every run. Each completed run must save its config, checkpoint, validation threshold, predictions, and metrics before moving on. The frozen T4 budget is about 40 minutes per run (roughly 6 GPU-hours total); disconnects should resume from verified checkpoints, not silently restart or mix versions.

In [ ]:
from importlib.util import find_spec
if find_spec('embeded.train') is None or find_spec('embeded.evaluate') is None:
    raise RuntimeError('Phase 2 runner modules are not implemented in this checkout.')
runs = [(c, seed) for c in ('C1','C2','C3') for seed in (13,14,15)]
for condition, seed in runs:
    print(f'\n===== TRAIN {condition} seed={seed} =====', flush=True)
    subprocess.run([sys.executable, '-m', 'embeded.train', '--condition', condition, '--seed', str(seed)], check=True)
    print(f'===== EVALUATE {condition} seed={seed} =====', flush=True)
    subprocess.run([sys.executable, '-m', 'embeded.evaluate', '--condition', condition, '--seed', str(seed)], check=True)
    hf_main(['push', '--if-configured'])  # checkpoint each completed run
print('All nine train/evaluate commands finished.')

## 7. Review results, record deviations, and checkpoint

Before reporting: verify all nine run artifacts and C0; check per-seed F1, precision, recall, MAP@R, and validation-selected thresholds; aggregate mean and spread without selecting the best seed. The test split is for final reporting only. Investigate implausible results as possible harness errors (pair ordering, labels, metrics, checkpoint, threshold code); never tune on test outcomes. Record outcomes and deviations in `report/measurements.md` and append actual completed work to the execution log in `PHASE2_PLAN.md`. Generalisation on BCB s′, UMAP, the demo, and paper/deck work are Phase 3+ and are not done here.

In [ ]:
from scripts.hf_artifacts import main as hf_main
hf_main(['push', '--if-configured', '--include-report'])
print('Review and commit the report and code through the repository workflow; keep large generated weights outside Git.')